In [5]:
from llava.model.builder import load_pretrained_model
import argparse
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path

model_path = "llava-fastvithd_0.5b_stage3"
model_name = get_model_name_from_path(model_path)

parser = argparse.ArgumentParser()
parser.add_argument("--model-path", type=str, default="./llava-v1.5-13b")
parser.add_argument("--model-base", type=str, default=None)
parser.add_argument("--image-file", type=str, default=None, help="location of image file")
parser.add_argument("--prompt", type=str, default="Describe the image.", help="Prompt for VLM.")
parser.add_argument("--conv-mode", type=str, default="mistral")
parser.add_argument("--temperature", type=float, default=0.2)
parser.add_argument("--top_p", type=float, default=None)
parser.add_argument("--num_beams", type=int, default=1)
args, unknown = parser.parse_known_args()

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path, args.model_base, model_name, device="cuda"
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [15]:
device = "cuda"

In [6]:
for name, module in model.named_modules():
    if "vision" in name.lower():
        print("VISION:", name)
    if "model" in name.lower():
        print("LLM:", name)

LLM: model
LLM: model.embed_tokens
LLM: model.layers
LLM: model.layers.0
LLM: model.layers.0.self_attn
LLM: model.layers.0.self_attn.q_proj
LLM: model.layers.0.self_attn.k_proj
LLM: model.layers.0.self_attn.v_proj
LLM: model.layers.0.self_attn.o_proj
LLM: model.layers.0.mlp
LLM: model.layers.0.mlp.gate_proj
LLM: model.layers.0.mlp.up_proj
LLM: model.layers.0.mlp.down_proj
LLM: model.layers.0.mlp.act_fn
LLM: model.layers.0.input_layernorm
LLM: model.layers.0.post_attention_layernorm
LLM: model.layers.1
LLM: model.layers.1.self_attn
LLM: model.layers.1.self_attn.q_proj
LLM: model.layers.1.self_attn.k_proj
LLM: model.layers.1.self_attn.v_proj
LLM: model.layers.1.self_attn.o_proj
LLM: model.layers.1.mlp
LLM: model.layers.1.mlp.gate_proj
LLM: model.layers.1.mlp.up_proj
LLM: model.layers.1.mlp.down_proj
LLM: model.layers.1.mlp.act_fn
LLM: model.layers.1.input_layernorm
LLM: model.layers.1.post_attention_layernorm
LLM: model.layers.2
LLM: model.layers.2.self_attn
LLM: model.layers.2.self_attn

In [7]:
vision_params = 0
llm_params = 0

for name, param in model.named_parameters():
    if "vision_tower" in name:
        vision_params += param.numel()
    else:
        llm_params += param.numel()

print(f"Vision params: {vision_params/1e6:.2f}M")
print(f"LLM params: {llm_params/1e6:.2f}M")

Vision params: 125.07M
LLM params: 497.33M


In [8]:
for name, param in model.named_parameters():
    if "mm_projector" in name:
        print(name, param.shape)

model.mm_projector.0.weight torch.Size([896, 3072])
model.mm_projector.0.bias torch.Size([896])
model.mm_projector.2.weight torch.Size([896, 896])
model.mm_projector.2.bias torch.Size([896])


In [9]:
model

LlavaQwen2ForCausalLM(
  (model): LlavaQwen2Model(
    (embed_tokens): Embedding(151646, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_e

In [6]:
vision_model = model.model.vision_tower
print(vision_model)

MobileCLIPVisionTower(
  (vision_tower): MCi(
    (model): FastViT(
      (patch_embed): Sequential(
        (0): MobileOneBlock(
          (se): Identity()
          (activation): GELU(approximate='none')
          (reparam_conv): Conv2d(3, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        )
        (1): MobileOneBlock(
          (se): Identity()
          (activation): GELU(approximate='none')
          (reparam_conv): Conv2d(96, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=96)
        )
        (2): MobileOneBlock(
          (se): Identity()
          (activation): GELU(approximate='none')
          (reparam_conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
        )
      )
      (network): ModuleList(
        (0): Sequential(
          (0): RepMixerBlock(
            (token_mixer): RepMixer(
              (reparam_conv): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=96)
            )
            (convffn): ConvFFN(
  

In [7]:
llm_model = model.model
print(llm_model)

LlavaQwen2Model(
  (embed_tokens): Embedding(151646, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
  (vision_tower): MobileCLIPVisionTower(
    (vi

In [8]:
fastvit = model.model.vision_tower.vision_tower.model
print(fastvit)

FastViT(
  (patch_embed): Sequential(
    (0): MobileOneBlock(
      (se): Identity()
      (activation): GELU(approximate='none')
      (reparam_conv): Conv2d(3, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (1): MobileOneBlock(
      (se): Identity()
      (activation): GELU(approximate='none')
      (reparam_conv): Conv2d(96, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=96)
    )
    (2): MobileOneBlock(
      (se): Identity()
      (activation): GELU(approximate='none')
      (reparam_conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    )
  )
  (network): ModuleList(
    (0): Sequential(
      (0): RepMixerBlock(
        (token_mixer): RepMixer(
          (reparam_conv): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=96)
        )
        (convffn): ConvFFN(
          (conv): Sequential(
            (conv): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96, bias=False)
            (bn): 

In [9]:
from safetensors.torch import load_file

weights = load_file("llava-fastvithd_0.5b_stage3/model.safetensors")

print(type(weights))
print(len(weights))  # number of tensors

# see some keys
for i, key in enumerate(weights.keys()):
    print(key, weights[key].shape)
    if i > 10:
        break

<class 'dict'>
924
lm_head.weight torch.Size([151936, 896])
model.embed_tokens.weight torch.Size([151936, 896])
model.layers.0.input_layernorm.weight torch.Size([896])
model.layers.0.mlp.down_proj.weight torch.Size([896, 4864])
model.layers.0.mlp.gate_proj.weight torch.Size([4864, 896])
model.layers.0.mlp.up_proj.weight torch.Size([4864, 896])
model.layers.0.post_attention_layernorm.weight torch.Size([896])
model.layers.0.self_attn.k_proj.bias torch.Size([128])
model.layers.0.self_attn.k_proj.weight torch.Size([128, 896])
model.layers.0.self_attn.o_proj.weight torch.Size([896, 896])
model.layers.0.self_attn.q_proj.bias torch.Size([896])
model.layers.0.self_attn.q_proj.weight torch.Size([896, 896])


In [10]:
import os
size = os.path.getsize("llava-fastvithd_0.5b_stage3/model.safetensors") / (1024**3)
print(f"{size:.2f} GB")

1.41 GB


In [11]:
import json

with open("llava-fastvithd_0.5b_stage3/config.json") as f:
    config = json.load(f)

print(config)

{'_name_or_path': './llava-v1.5-13b', 'architectures': ['LlavaQwen2ForCausalLM'], 'attention_dropout': 0.0, 'bos_token_id': 151643, 'eos_token_id': 151645, 'freeze_mm_mlp_adapter': False, 'hidden_act': 'silu', 'hidden_size': 896, 'image_aspect_ratio': 'pad', 'image_grid_pinpoints': None, 'initializer_range': 0.02, 'intermediate_size': 4864, 'max_position_embeddings': 32768, 'max_window_layers': 24, 'mm_hidden_size': 3072, 'mm_patch_merge_type': 'flat', 'mm_projector_lr': None, 'mm_projector_type': 'mlp2x_gelu', 'mm_use_im_patch_token': False, 'mm_use_im_start_end': False, 'mm_vision_select_feature': 'patch', 'mm_vision_select_layer': -2, 'mm_vision_tower': 'mobileclip_l_1024', 'model_type': 'llava_qwen2', 'num_attention_heads': 14, 'num_hidden_layers': 24, 'num_key_value_heads': 2, 'rms_norm_eps': 1e-06, 'rope_theta': 1000000.0, 'sliding_window': 32768, 'tie_word_embeddings': True, 'tokenizer_model_max_length': 8192, 'tokenizer_padding_side': 'right', 'torch_dtype': 'bfloat16', 'transf

In [12]:
print(config["hidden_size"]), print(config["num_hidden_layers"]), print(config["num_attention_heads"]), print(config["model_type"])

896
24
14
llava_qwen2


(None, None, None, None)

In [13]:
text = "Hello bhai kya haal hai"
tokens = tokenizer(text)

print(tokens)
print(tokenizer.decode(tokens["input_ids"]))

{'input_ids': [9707, 293, 25214, 595, 7755, 6386, 278, 46051], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}
Hello bhai kya haal hai


In [14]:
from safetensors.torch import load_file

weights = load_file("llava-fastvithd_0.5b_stage3/model.safetensors")

print(len(weights))

for k in list(weights.keys())[:20]:
    print(k, weights[k].shape)

924
lm_head.weight torch.Size([151936, 896])
model.embed_tokens.weight torch.Size([151936, 896])
model.layers.0.input_layernorm.weight torch.Size([896])
model.layers.0.mlp.down_proj.weight torch.Size([896, 4864])
model.layers.0.mlp.gate_proj.weight torch.Size([4864, 896])
model.layers.0.mlp.up_proj.weight torch.Size([4864, 896])
model.layers.0.post_attention_layernorm.weight torch.Size([896])
model.layers.0.self_attn.k_proj.bias torch.Size([128])
model.layers.0.self_attn.k_proj.weight torch.Size([128, 896])
model.layers.0.self_attn.o_proj.weight torch.Size([896, 896])
model.layers.0.self_attn.q_proj.bias torch.Size([896])
model.layers.0.self_attn.q_proj.weight torch.Size([896, 896])
model.layers.0.self_attn.v_proj.bias torch.Size([128])
model.layers.0.self_attn.v_proj.weight torch.Size([128, 896])
model.layers.1.input_layernorm.weight torch.Size([896])
model.layers.1.mlp.down_proj.weight torch.Size([896, 4864])
model.layers.1.mlp.gate_proj.weight torch.Size([4864, 896])
model.layers.1.

In [15]:
import torch
from PIL import Image
from torchvision import transforms

# Load image
image = Image.open("GOT.jpg").convert("RGB")

# Preprocess image: resize, to tensor, normalize
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),  # ya jo size model expect karta hai
    transforms.ToTensor(),          # PIL -> [C, H, W] tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet style normalization
                         std=[0.229, 0.224, 0.225])
])

image_tensor = preprocess(image).unsqueeze(0).to("cuda").half()  # [1, 3, H, W]

# Forward pass through vision tower
vision_features = model.model.vision_tower(image_tensor)

# Project to LLM embedding space
projected_features = model.model.mm_projector(vision_features)

print(projected_features.shape)  # should be [1, seq_len, 896]

torch.Size([1, 16, 896])


In [16]:
# pseudo code
prompt = "Describe this image:"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
output_ids = model.generate(
    input_ids,
            images=image_tensor.unsqueeze(0).half(),
            image_sizes=[image.size],
            do_sample=True if args.temperature > 0 else False,
            temperature=args.temperature,
            top_p=args.top_p,
            num_beams=args.num_beams,
            max_new_tokens=256,
            use_cache=True)
# outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 The image is a photograph of a person. The person is wearing a white shirt. The shirt has a small logo on the left chest area. The logo is a red circle with a white line through it. The person is standing with their arms at their sides. The background of the image is blurred. The person's face is not visible in the image. The image is in color.


In [17]:
import torch

class IdentitySE(torch.nn.Module):
    def forward(self, x):
        return x

def replace_seblock(module):
    for name, child in module.named_children():
        if child.__class__.__name__ == "SEBlock":
            setattr(module, name, IdentitySE())
        else:
            replace_seblock(child)

vision_model = model.model.vision_tower.vision_tower.model
replace_seblock(vision_model)

In [18]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

vision_model.eval()
vision_model = vision_model.to(device)
# vision_model = vision_model.half()  # Comment out for now

image_tensor = image_processor(image, return_tensors="pt")["pixel_values"]
image_tensor = image_tensor.to(device)
image_tensor = image_tensor.half()  # Comment out for now

torch.onnx.export(
    vision_model,
    image_tensor,
    "vision.onnx",
    opset_version=14,
    input_names=["image"],
    output_names=["image_embeds"],
    dynamic_axes={"image": {0: "batch"}, "image_embeds": {0: "batch"}}
)

In [19]:
llm_model = model.model
llm_model.to("cuda").eval()

LlavaQwen2Model(
  (embed_tokens): Embedding(151646, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
  (vision_tower): MobileCLIPVisionTower(
    (vi

In [20]:
model

LlavaQwen2ForCausalLM(
  (model): LlavaQwen2Model(
    (embed_tokens): Embedding(151646, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_e

In [21]:
model.model.vision_tower

MobileCLIPVisionTower(
  (vision_tower): MCi(
    (model): FastViT(
      (patch_embed): Sequential(
        (0): MobileOneBlock(
          (se): Identity()
          (activation): GELU(approximate='none')
          (reparam_conv): Conv2d(3, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        )
        (1): MobileOneBlock(
          (se): Identity()
          (activation): GELU(approximate='none')
          (reparam_conv): Conv2d(96, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=96)
        )
        (2): MobileOneBlock(
          (se): Identity()
          (activation): GELU(approximate='none')
          (reparam_conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
        )
      )
      (network): ModuleList(
        (0): Sequential(
          (0): RepMixerBlock(
            (token_mixer): RepMixer(
              (reparam_conv): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=96)
            )
            (convffn): ConvFFN(
  

In [22]:
for name, param in llm_model.layers[0].self_attn.named_parameters():
    print(name, param.shape)

q_proj.weight torch.Size([896, 896])
q_proj.bias torch.Size([896])
k_proj.weight torch.Size([128, 896])
k_proj.bias torch.Size([128])
v_proj.weight torch.Size([128, 896])
v_proj.bias torch.Size([128])
o_proj.weight torch.Size([896, 896])


In [23]:
model.model

LlavaQwen2Model(
  (embed_tokens): Embedding(151646, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
  (vision_tower): MobileCLIPVisionTower(
    (vi

In [24]:
class DummyAttention(torch.nn.Module):
    def forward(self, hidden_states, *args, **kwargs):
        return hidden_states, None, None

for layer in llm_model.layers:
    layer.self_attn = DummyAttention()
    
class DummyRotary(torch.nn.Module):
    def forward(self, x, *args, **kwargs):
        return x

for layer in llm_model.layers:
    layer.self_attn.rotary_emb = DummyRotary()

for name, module in llm_model.named_modules():
    if "self_attn" in name:
        parent = llm_model
        parts = name.split(".")
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], DummyAttention())

In [25]:
model.model

LlavaQwen2Model(
  (embed_tokens): Embedding(151646, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): DummyAttention(
        (rotary_emb): DummyAttention()
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
  (vision_tower): MobileCLIPVisionTower(
    (vision_tower): MCi(
      (model): FastViT(
        (patch_embed): Sequential(
          (0): MobileOneBlock(
            (se): Identity()
            (activation): GELU(approximate='none')
            (reparam_conv): Conv2d(3, 96, kernel_size=(3, 

In [26]:
model

LlavaQwen2ForCausalLM(
  (model): LlavaQwen2Model(
    (embed_tokens): Embedding(151646, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): DummyAttention(
          (rotary_emb): DummyAttention()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbedding()
    (vision_tower): MobileCLIPVisionTower(
      (vision_tower): MCi(
        (model): FastViT(
          (patch_embed): Sequential(
            (0): MobileOneBlock(
              (se): Identity()
              (activation):

In [27]:
class CleanWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model  # 👈 direct base model

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=False
        )
        return outputs[0]

In [30]:
wrapped_model = CleanWrapper(llm_model)

In [34]:
# Dummy inputs
batch_size = 1
seq_len = 4
input_ids = torch.randint(0, 151646, (batch_size, seq_len), dtype=torch.long, device=device)
attention_mask = torch.ones((batch_size, seq_len), dtype=torch.long, device=device)

original_layers = llm_model.layers

for i in range(1, len(original_layers) + 1):
    print(f"Testing {i} layers")

    llm_model.layers = torch.nn.ModuleList(original_layers[:i])

    try:
        torch.onnx.export(
            wrapped_model,
            (input_ids, attention_mask),
            f"test_{i}.onnx",
            opset_version=17,
            dynamo=False,
            do_constant_folding=False
        )
        print(f"✅ Success at layer {i}")
    except Exception as e:
        print(f"❌ Failed at layer {i}: {e}")
        break

# restore model
llm_model.layers = original_layers

Testing 1 layers
❌ Failed at layer 1: invalid unordered_map<K, T> key


In [38]:
layer = model.model.layers[0]

x = llm_model.embed_tokens(input_ids)

try:
    out = layer(x, attention_mask=attention_mask)
    print("Layer forward success")
except Exception as e:
    print("Layer forward failed:", e)

Layer forward failed: too many values to unpack (expected 2)


In [46]:
model.model

LlavaQwen2Model(
  (embed_tokens): Embedding(151646, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
  (vision_tower): MobileCLIPVisionTower(
    (vi

In [47]:
layer = model.model.layers[0]

print(layer)

Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=896, out_features=896, bias=True)
    (k_proj): Linear(in_features=896, out_features=128, bias=True)
    (v_proj): Linear(in_features=896, out_features=128, bias=True)
    (o_proj): Linear(in_features=896, out_features=896, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)


In [10]:
llm_model2 = model.model

In [11]:
llm_model2.vision_tower = None
llm_model2.mm_projector = None

In [12]:
llm_model2

LlavaQwen2Model(
  (embed_tokens): Embedding(151646, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
  (vision_tower): None
  (mm_projector): None
)

In [55]:
class StaticRotary(torch.nn.Module):
    def forward(self, x, *args, **kwargs):
        b, seq_len, dim = x.shape
        
        cos = torch.ones((1, seq_len, dim), device=x.device, dtype=x.dtype)
        sin = torch.zeros((1, seq_len, dim), device=x.device, dtype=x.dtype)
        
        return cos, sin

In [56]:
llm_model2.rotary_emb = StaticRotary()

In [57]:
llm_model2

LlavaQwen2Model(
  (embed_tokens): Embedding(151646, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): Qwen2Attention(
        (q_proj): Linear(in_features=896, out_features=896, bias=True)
        (k_proj): Linear(in_features=896, out_features=128, bias=True)
        (v_proj): Linear(in_features=896, out_features=128, bias=True)
        (o_proj): Linear(in_features=896, out_features=896, bias=False)
      )
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): StaticRotary()
  (vision_tower): None
  (mm_projector): None
)

In [58]:
llm_model2.config.use_cache = False

In [18]:
class DummyAttention(torch.nn.Module):
    def forward(self, hidden_states, *args, **kwargs):
        return hidden_states, None  # match unpacking

In [19]:
for layer in llm_model2.layers:
    layer.self_attn = DummyAttention()

In [20]:
llm_model2

LlavaQwen2Model(
  (embed_tokens): Embedding(151646, 896)
  (layers): ModuleList(
    (0-23): 24 x Qwen2DecoderLayer(
      (self_attn): DummyAttention()
      (mlp): Qwen2MLP(
        (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
        (up_proj): Linear(in_features=896, out_features=4864, bias=False)
        (down_proj): Linear(in_features=4864, out_features=896, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
    )
  )
  (norm): Qwen2RMSNorm((896,), eps=1e-06)
  (rotary_emb): Qwen2RotaryEmbedding()
  (vision_tower): None
  (mm_projector): None
)

In [65]:
try:
    cos, sin = llm_model2.rotary_emb(torch.randn(1,4,896).to(device))
    print("Rotary OK")
except Exception as e:
    print("Rotary failed:", e)

Rotary OK


In [66]:
x = llm_model2.embed_tokens(input_ids)

out = llm_model2.layers[0](x, attention_mask=attention_mask)
print("Layer works ✅")

Layer works ✅


In [67]:
torch.onnx.export(
    llm_model2,
    (input_ids, attention_mask),
    "llava_qwen2_clean.onnx",

    input_names=["input_ids", "attention_mask"],
    output_names=["last_hidden_state"],

    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "attention_mask": {0: "batch", 1: "seq"},
        "last_hidden_state": {0: "batch", 1: "seq"}
    },

    opset_version=17,
    do_constant_folding=True,
    export_params=True
)

print("✅ ONNX export successful!")

TypeError: Qwen2Model.forward() got multiple values for argument 'use_cache'

In [13]:
from transformers.models.qwen2 import modeling_qwen2

def dummy_causal_mask(*args, **kwargs):
    return None

modeling_qwen2.create_causal_mask = dummy_causal_mask

In [21]:
import torch
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(False)

# =========================
# 1. Wrapper (VERY IMPORTANT)
# =========================
class ONNXWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        return self.model(
            input_ids=input_ids,
            attention_mask=None,   # mask disable
            use_cache=False,
            return_dict=False
        )[0]


wrapped_model = ONNXWrapper(llm_model2).to(device)
wrapped_model.eval()

input_ids = torch.randint(
    0, 151646, (1, 16),
    dtype=torch.long,
    device=device
)

torch.onnx.export(
    wrapped_model,
    (input_ids,),
    "llava_qwen2_clean.onnx",

    input_names=["input_ids"],
    output_names=["last_hidden_state"],

    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq"},
        "last_hidden_state": {0: "batch", 1: "seq"}
    },

    opset_version=17,
    do_constant_folding=True
)

print("✅ ONNX export successful!")

✅ ONNX export successful!


In [ ]:
import onnx

model = onnx.load("llava_qwen2_clean.onnx")
onnx.checker.check_model(model)

print("✅ Model is valid")

In [1]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("llava_qwen2_clean.onnx")

input_ids = np.random.randint(0, 151646, (1, 16)).astype(np.int64)

outputs = session.run(None, {
    "input_ids": input_ids
})

print(outputs[0].shape)

ImportError: DLL load failed while importing onnxruntime_pybind11_state: A dynamic link library (DLL) initialization routine failed.

In [71]:
wrapped_model

ONNXWrapper(
  (model): LlavaQwen2Model(
    (embed_tokens): Embedding(151646, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): DummyAttention()
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): StaticRotary()
    (vision_tower): None
    (mm_projector): None
    (self_attn): DummyAttention()
  )
)

In [51]:
layer = model.model.layers[10]

print(layer)

Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=896, out_features=896, bias=True)
    (k_proj): Linear(in_features=896, out_features=128, bias=True)
    (v_proj): Linear(in_features=896, out_features=128, bias=True)
    (o_proj): Linear(in_features=896, out_features=896, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)


In [29]:
with torch.no_grad():
    out = llm_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        return_dict=False
    )
    print(type(out), len(out))

NameError: name 'attention_mask' is not defined

In [ ]:
llm_model.layers = llm_model.layers[:1]

In [ ]:
class LLMWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=False
        )
        return outputs[0]

llm_model.config.attn_implementation = "eager"
llm_model.layers = llm_model.layers[:1]
wrapped_model = LLMWrapper(llm_model)

In [ ]:
onnx_program = torch.onnx.export(
        wrapped_model,
        (input_ids, attention_mask),          # positional tuple
        "llm.onnx",
        dynamo=False,                          # ← yeh important hai
        opset_version=18,                     # 18 best hai Qwen2 ke liye (19 bhi try kar sakte ho)
        input_names=["input_ids", "attention_mask"],
        output_names=["last_hidden_state"],
        dynamic_axes={
            "input_ids": {0: "batch_size", 1: "sequence_length"},
            "attention_mask": {0: "batch_size", 1: "sequence_length"},
            "last_hidden_state": {0: "batch_size", 1: "sequence_length"}
        },
        do_constant_folding=False,
        verbose=False
    )
print("✅ LLM successfully exported as llm.onnx using dynamo=True")

In [ ]:
import torch

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

# llm_model = model.model
model.config.attn_implementation = "eager"
llm_model.eval()
llm_model = llm_model.to(device)

# Wrapper (same as before)
class LLMWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=False   # 🔥 change
        )
        return outputs[0]  # last_hidden_state

wrapped_model = LLMWrapper(llm_model)

# Dummy inputs
batch_size = 1
seq_len = 4
input_ids = torch.randint(0, 151646, (batch_size, seq_len), dtype=torch.long)
attention_mask = torch.ones((batch_size, seq_len), dtype=torch.long)

# Vision tower bypass
if hasattr(llm_model, "vision_tower"):
    llm_model.vision_tower = None

try:
    # NEW RECOMMENDED WAY
    onnx_program = torch.onnx.export(
        wrapped_model,
        (input_ids, attention_mask),          # positional tuple
        "llm.onnx",
        dynamo=False,                          # ← yeh important hai
        opset_version=18,                     # 18 best hai Qwen2 ke liye (19 bhi try kar sakte ho)
        input_names=["input_ids", "attention_mask"],
        output_names=["last_hidden_state"],
        dynamic_axes={
            "input_ids": {0: "batch_size", 1: "sequence_length"},
            "attention_mask": {0: "batch_size", 1: "sequence_length"},
            "last_hidden_state": {0: "batch_size", 1: "sequence_length"}
        },
        do_constant_folding=False,
        verbose=False
    )
    print("✅ LLM successfully exported as llm.onnx using dynamo=True")
    
except Exception as e:
    print("Error:", e)
    raise

finally:
    if 'original_vision' in locals():
        llm_model.vision_tower = original_vision

In [ ]:
decoder_layer = llm_model.layers[0]

def print_module(module, prefix=""):
    """
    Recursively print all submodules and their parameters.
    """
    print(f"{prefix}{module.__class__.__name__}")
    
    # Print parameters of this module
    for name, param in module.named_parameters(recurse=False):
        print(f"  {prefix}Param: {name} | shape: {param.shape}")
    
    # Recursively print children
    for name, child in module.named_children():
        print_module(child, prefix + name + ".")

# Run the print for the first decoder layer
print_module(decoder_layer)

In [ ]:
# Model device and dtype
for name, param in llm_model.named_parameters():
    print(name, param.device, param.dtype)

In [ ]:
for i, layer in enumerate(llm_model.layers):
    for sub_name, sub_module in layer.named_modules():
        if any(not isinstance(p, torch.Tensor) for p in sub_module.parameters(recurse=False)):
            print(f"Layer {i} module {sub_name} has non-tensor parameters!")

In [ ]:
for i, layer in enumerate(llm_model.layers):
    if hasattr(layer.self_attn, "rotary_emb"):
        print(f"Layer {i} has rotary_emb")
    else:
        print(f"Layer {i} MISSING rotary_emb")

In [ ]:
for name, module in llm_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(name, module.weight.shape)
        if module.bias is not None:
            print(name, module.bias.shape)

In [ ]:
print(llm_model.layers[0].self_attn.forward.__code__.co_varnames)

In [ ]:
for name, module in llm_model.vision_tower.named_modules():
    print(name, type(module))

In [ ]:
batch_size = 1
seq_len = 128
hidden_size = 896
num_heads = 14
head_dim = hidden_size // num_heads  # 64

device = "cuda" if torch.cuda.is_available() else "cpu"

# Dummy input
dummy_x = torch.randn(batch_size, seq_len, hidden_size, device=device).half()
dummy_attention_mask = torch.ones(batch_size, seq_len, dtype=torch.long, device=device)

# Manual rotary embeddings
cos = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device).half()
sin = torch.randn(batch_size, num_heads, seq_len, head_dim, device=device)
dummy_position_embeddings = (cos, sin)

# Forward
decoder_layer = llm_model.layers[0].to(device).half()

with torch.no_grad():
    out = decoder_layer.self_attn(
        dummy_x,
        attention_mask=dummy_attention_mask,
        position_embeddings=dummy_position_embeddings,
        use_cache=False
    )

print("Forward output shape:", out.shape)

In [ ]:
x = embed
for i, layer in enumerate(model.layers):
    try:
        x = layer(x)  # forward through one decoder layer
        print(f"✅ Layer {i} forward ok, output shape: {x.shape}")
    except Exception as e:
        print(f"❌ Layer {i} forward failed:", e)
        break

In [ ]:
# Define dimensions
vocab_size = 151646  
batch_size = 1       # usually 1 for export
seq_len = 128  

In [ ]:
dummy_input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
dummy_attention_mask = torch.ones((batch_size, seq_len))
dummy_image = torch.randn((batch_size, 3, 224, 224))

In [ ]:
device = "cpu"
model.to(device)
dummy_input_ids = dummy_input_ids.to(device)
dummy_attention_mask = dummy_attention_mask.to(device)
dummy_image = dummy_image.to(device)

In [ ]:
input_names = ["input_ids", "attention_mask", "image"]
output_names = ["logits"]  # or whatever your model outputs

dynamic_axes = {
    "input_ids": {0: "batch_size", 1: "seq_len"},
    "attention_mask": {0: "batch_size", 1: "seq_len"},
    "image": {0: "batch_size"},
    "logits": {0: "batch_size", 1: "seq_len"}
}

In [ ]:
with torch.no_grad():
    traced_model = torch.jit.trace(
        model,
        (dummy_input_ids, dummy_attention_mask, dummy_image),
        strict=False
    )

In [ ]:
torch.onnx.export(
    model,
    (dummy_input_ids, dummy_attention_mask, dummy_image),  # tuple of inputs
    "llavaqwen2.onnx",
    input_names=input_names,
    output_names=output_names,
    opset_version=17,          # recommended for transformers + vision
    do_constant_folding=True,  # optimize model
    dynamic_axes=dynamic_axes
)

print("✅ ONNX export complete!")

In [ ]:
import torch

class LLMWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model.eval()

    def forward(self, input_ids, attention_mask):
        # Ensure inputs are on the same device as model
        device = next(self.model.parameters()).device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False
        )
        return outputs.logits

llm_wrapper = LLMWrapper(llm_model.to("cuda"))

dummy_input_ids = torch.randint(0, 151646, (1, 128)).to("cuda")
dummy_attention_mask = torch.ones((1, 128), dtype=torch.long).to("cuda")

with torch.no_grad():
    traced_model = torch.jit.trace(
        llm_wrapper,
        (dummy_input_ids, dummy_attention_mask),
        strict=False
    )

scripted_model = torch.jit.script(llm_wrapper)
scripted_model.save("llm_script.pt")

# Then export traced model to ONNX
torch.onnx.export(
    traced_model,
    (dummy_input_ids, dummy_attention_mask),
    "llm.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids": {0: "batch", 1: "seq_len"},
        "attention_mask": {0: "batch", 1: "seq_len"},
        "logits": {0: "batch", 1: "seq_len"},
    },
    opset_version=15,
    do_constant_folding=True
)

In [ ]:
wrapped_model = CleanWrapper(llm_model)

torch.onnx.export(
    wrapped_model,
    (input_ids, attention_mask),
    "clean.onnx",
    opset_version=16,
    dynamo=False,
    do_constant_folding=False
)